In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from pathlib import Path

df = pd.read_csv("Day12_Used_Car_Preprocessing_Dataset.csv")
print("Shape:", df.shape)
display(df.head())
display(df.dtypes)

Shape: (320, 15)


,Car_ID,Brand,Year,Mileage_Km,Engine_CC,Power_BHP,Fuel_Type,Transmission,City,Seller_Type,Condition,Previous_Owners,Accidents_Reported,Service_Score,Resale_Price_Lakh
0,CAR0001,Skoda,2021,69708,1152,128.8,Diesel,Manual,Lucknow,Individual,Good,1,0,72,6.38
1,CAR0002,Toyota,2020,88881,903,146.5,Diesel,Automatic,Chandigarh,Individual,Good,1,0,87,4.83
2,CAR0003,Volkswagen,2021,43646,1446,185.9,Diesel,Automatic,Hyderabad,Individual,Very Good,2,0,90,7.30
3,CAR0004,Tata,2019,70847,2069,148.8,Petrol,Manual,Lucknow,Individual,Excellent,3,0,66,3.82
4,CAR0005,Tata,2016,101228,1657,206.0,Petrol,Automatic,Ahmedabad,Dealer,Very Good,2,0,84,1.93


Car_ID                    str
Brand                     str
Year                    int64
Mileage_Km              int64
Engine_CC               int64
Power_BHP             float64
Fuel_Type                 str
Transmission              str
City                      str
Seller_Type               str
Condition                 str
Previous_Owners         int64
Accidents_Reported      int64
Service_Score           int64
Resale_Price_Lakh     float64
dtype: object

In [2]:
print("Missing values:")
display(df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())
display(df.describe(include="all").T)

Missing values:


Car_ID                0
Brand                 0
Year                  0
Mileage_Km            0
Engine_CC             0
Power_BHP             0
Fuel_Type             0
Transmission          0
City                  0
Seller_Type           0
Condition             0
Previous_Owners       0
Accidents_Reported    0
Service_Score         0
Resale_Price_Lakh     0
dtype: int64

Duplicate rows: 0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Car_ID,320,320,CAR0001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Brand,320,10,Volkswagen,39,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Year,320.0,NaN,NaN,NaN,2019.5375,3.341367,2014.0,2017.0,2020.0,2022.0,2025.0
Mileage_Km,320.0,NaN,NaN,NaN,74110.203125,38885.260771,700.0,46323.25,72718.5,97951.5,320000.0
Engine_CC,320.0,NaN,NaN,NaN,1346.703125,543.40816,600.0,1004.75,1303.0,1635.25,5000.0
Power_BHP,320.0,NaN,NaN,NaN,150.489688,36.665353,51.4,128.45,150.75,171.475,390.0
Fuel_Type,320,4,Petrol,182,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Transmission,320,2,Manual,197,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,320,10,Lucknow,43,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Seller_Type,320,3,Individual,167,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Outlier Detection

In [3]:
raw_num = df.select_dtypes(include=np.number).drop(columns=["Resale_Price_Lakh"])
outlier_counts = {}
for c in raw_num.columns:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    outlier_counts[c] = int(((df[c] < lo) | (df[c] > hi)).sum())

outlier_table = pd.DataFrame({
    "Feature": list(outlier_counts.keys()),
    "IQR_Outliers": list(outlier_counts.values())
})
display(outlier_table)

,Feature,IQR_Outliers
0,Year,0
1,Mileage_Km,2
2,Engine_CC,6
3,Power_BHP,7
4,Previous_Owners,14
5,Accidents_Reported,63
6,Service_Score,0


Feature/Target Separation

In [4]:
df_clean = df.drop_duplicates().copy()

X = df_clean.drop(columns=["Resale_Price_Lakh", "Car_ID"])
y = df_clean["Resale_Price_Lakh"]

print("Cleaned shape:", df_clean.shape)
print("X shape:", X.shape)
print("y shape:", y.shape)

Cleaned shape: (320, 15)
X shape: (320, 13)
y shape: (320,)


Train/Test Split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 256
Testing rows: 64


Preprocessing Pipelines

In [6]:
class IQRClipper(BaseEstimator, TransformerMixin):
    def __init__(self, factor=1.5):
        self.factor = factor

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.q1_ = np.nanpercentile(X, 25, axis=0)
        self.q3_ = np.nanpercentile(X, 75, axis=0)
        iqr = self.q3_ - self.q1_
        self.lower_ = self.q1_ - self.factor * iqr
        self.upper_ = self.q3_ + self.factor * iqr
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lower_, self.upper_)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)

num_cols = X.select_dtypes(include=np.number).columns.tolist()
nominal_cols = ["Brand", "Fuel_Type", "Transmission", "City", "Seller_Type"]
ordinal_cols = ["Condition"]

ordinal_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(
        categories=[["Poor", "Fair", "Good", "Excellent"]],
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("iqr_clip", IQRClipper()),
    ("scale", StandardScaler())
])

nominal_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("nom", nominal_pipe, nominal_cols),
    ("ord", ordinal_pipe, ordinal_cols)
])

print("Numerical:", num_cols)
print("Nominal:", nominal_cols)
print("Ordinal:", ordinal_cols)

Numerical: ['Year', 'Mileage_Km', 'Engine_CC', 'Power_BHP', 'Previous_Owners', 'Accidents_Reported', 'Service_Score']
Nominal: ['Brand', 'Fuel_Type', 'Transmission', 'City', 'Seller_Type']
Ordinal: ['Condition']


Fit Only on Training Data

In [7]:
X_train_p = preprocessor.fit_transform(X_train)
X_test_p = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
X_train_p = pd.DataFrame(X_train_p, columns=feature_names, index=X_train.index)
X_test_p = pd.DataFrame(X_test_p, columns=feature_names, index=X_test.index)

print("Processed train shape:", X_train_p.shape)
print("Processed test shape:", X_test_p.shape)
display(X_train_p.head())

Processed train shape: (256, 37)
Processed test shape: (64, 37)


,num__Year,num__Mileage_Km,num__Engine_CC,num__Power_BHP,num__Previous_Owners,num__Accidents_Reported,num__Service_Score,nom__Brand_Honda,nom__Brand_Hyundai,nom__Brand_Kia,...,nom__City_Hyderabad,nom__City_Jaipur,nom__City_Kochi,nom__City_Lucknow,nom__City_Mumbai,nom__City_Pune,nom__Seller_Type_Certified Dealer,nom__Seller_Type_Dealer,nom__Seller_Type_Individual,ord__Condition
132,-0.486391,-0.225883,-0.322882,0.304485,-0.755752,0.0,-0.454105,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,-1.0
317,0.725445,0.089371,-0.656431,-0.330444,-0.755752,0.0,-0.614672,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2.0
234,-1.395268,1.115798,-0.120529,0.420784,-0.755752,0.0,-0.534389,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0
312,1.028404,-0.195076,0.479859,0.414498,0.484456,0.0,0.188165,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,2.0
232,-1.092309,0.706680,0.786724,-0.437314,0.484456,0.0,0.107881,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-1.0


Verification

In [8]:
print("Training values numeric:", all(pd.api.types.is_numeric_dtype(t) for t in X_train_p.dtypes))
print("Test values numeric:", all(pd.api.types.is_numeric_dtype(t) for t in X_test_p.dtypes))
print("Missing processed train values:", int(X_train_p.isna().sum().sum()))
print("Missing processed test values:", int(X_test_p.isna().sum().sum()))

print("\nProcessed numerical-feature statistics:")
display(X_train_p[["num__Year", "num__Mileage_Km", "num__Engine_CC", "num__Power_BHP",
                   "num__Previous_Owners", "num__Accidents_Reported",
                   "num__Service_Score"]].describe().T)

Training values numeric: True
Test values numeric: True
Missing processed train values: 0
Missing processed test values: 0

Processed numerical-feature statistics:


,count,mean,std,min,25%,50%,75%,max
num__Year,256.0,-3.816392e-17,1.001959,-1.698227,-0.789350,0.119527,0.725445,1.634322
num__Mileage_Km,256.0,1.040834e-17,1.001959,-2.046919,-0.770053,-0.010386,0.688895,2.877317
num__Engine_CC,256.0,3.469447e-18,1.001959,-1.630394,-0.711467,-0.056043,0.692774,2.799136
num__Power_BHP,256.0,-7.077672e-16,1.001959,-2.597881,-0.642408,0.012166,0.661240,2.616712
num__Previous_Owners,256.0,-4.163336e-17,1.001959,-0.755752,-0.755752,-0.755752,0.484456,2.344768
num__Accidents_Reported,256.0,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
num__Service_Score,256.0,-6.938894e-18,1.001959,-1.738645,-0.875595,0.107881,0.750151,1.713556


Export Train, Test, and Combined Processed Data

In [9]:
outdir = Path("used_car_preprocessing")
outdir.mkdir(exist_ok=True)

train_out = X_train_p.copy()
train_out["Resale_Price_Lakh"] = y_train
train_out["Dataset_Split"] = "Train"

test_out = X_test_p.copy()
test_out["Resale_Price_Lakh"] = y_test
test_out["Dataset_Split"] = "Test"

combined = pd.concat([train_out, test_out]).sort_index()
combined.index.name = "Original_Row_Index"

train_out.to_csv(outdir / "used_car_preprocessed_train.csv", index=False)
test_out.to_csv(outdir / "used_car_preprocessed_test.csv", index=False)
combined.to_csv(outdir / "used_car_preprocessed_dataset.csv", index=True)

print("Saved successfully.")

Saved successfully.


End-to-End Verification with a Simple Model

In [10]:

model = LinearRegression()
model.fit(X_train_p, y_train)
pred = model.predict(X_test_p)

mae = mean_absolute_error(y_test, pred)
rmse = mean_squared_error(y_test, pred) ** 0.5
r2 = r2_score(y_test, pred)

print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²  : {r2:.4f}")

MAE : 1.2591
RMSE: 1.6081
R²  : 0.6662
